# 📐 Python Arrays — The Master Guide
### *From Zero to Interview-Ready*

---

> **Mental Model First:**
> An array is a **numbered row of mailboxes** — each box has an address (index) and holds one value.
> You can reach any box instantly by address — that's O(1) access.
> The three interview superpowers built on top of arrays are:
> **Two Pointer** (two fingers scanning the row), **Sliding Window** (a moving frame over the row),
> and **Prefix Sum** (a running total stored alongside the row).
> Every array interview problem is one of these three in disguise.

---

## 📋 Table of Contents

| # | Section |
|---|---|
| 1 | [What Is an Array? The Visual Model](#1) |
| 2 | [Creating / Setup](#2) |
| 3 | [The Core API — All Operations](#3) |
| 4 | [Decision Map — Two Pointer vs Sliding Window vs Prefix Sum](#4) |
| 5 | [Pattern 1: Two Pointer — Opposite Ends (LC 167)](#5) |
| 6 | [Pattern 2: Two Pointer — Same Direction / Fast-Slow (LC 26)](#6) |
| 7 | [Pattern 3: Fixed Sliding Window (LC 643)](#7) |
| 8 | [Pattern 4: Variable Sliding Window (LC 3)](#8) |
| 9 | [Pattern 5: Prefix Sum — Range Query (LC 560)](#9) |
| 10 | [Pattern 6: Product Array Without Self (LC 238)](#10) |
| 11 | [The Array Pattern Decision Map](#11) |
| 12 | [Interview Cheat Sheet](#12) |

<a id='1'></a>
## 1. 🏗️ What Is an Array? The Visual Model

---

```
                    THE MAILBOX ROW

  Index:    0      1      2      3      4      5
          ┌──────┬──────┬──────┬──────┬──────┬──────┐
  Value:  │  1   │  3   │  5   │  7   │  9   │  11  │
          └──────┴──────┴──────┴──────┴──────┴──────┘
            ▲                                  ▲
            L                                  R
         left ptr                          right ptr

  nums[0]  = 1    → O(1) — direct address lookup
  nums[-1] = 11   → O(1) — Python negative indexing from right
  nums[2:5]= [5,7,9] → O(k) slice — k elements copied

  TWO POINTER: two fingers scanning the row
  ┌──────┬──────┬──────┬──────┬──────┬──────┐
  │  1   │  3   │  5   │  7   │  9   │  11  │
  └──────┴──────┴──────┴──────┴──────┴──────┘
    L →                              ← R
  Fingers move toward each other. Stop when they meet.

  SLIDING WINDOW: a moving frame
  ┌──────┬──────┬──────┬──────┬──────┬──────┐
  │  1   │  3   │  5   │  7   │  9   │  11  │
  └──────┴──────┴──────┴──────┴──────┴──────┘
  [  L ════════ R  ]  →  slide right
  Frame grows/shrinks as it moves across the row.

  PREFIX SUM: a shadow array of running totals
  original: [  1  ,  3  ,  5  ,  7  ,  9  ,  11  ]
  prefix:   [  0  ,  1  ,  4  ,  9  , 16  ,  25  ,  36 ]
  Sum of any range [i..j] = prefix[j+1] - prefix[i]  → O(1)
```

---

### Why do these three patterns dominate?

**Two Pointer** turns O(n²) brute force (try all pairs) → O(n) by exploiting sort order or monotonicity.

**Sliding Window** turns O(n·k) brute force (recompute each window) → O(n) by adding the new element and dropping the old one.

**Prefix Sum** turns O(n) per query (loop and sum) → O(1) per query by pre-computing totals once.

<a id='2'></a>
## 2. 🔧 Creating / Setup

In [ ]:
from typing import List

# --- Empty array ---
arr = []
print("Empty:", arr)

# --- Literal ---
nums = [1, 3, 5, 7, 9, 11]
print("Literal:", nums)

# --- From range ---
r = list(range(6))          # [0, 1, 2, 3, 4, 5]
print("From range:", r)

# --- Filled with zeros (common for DP / prefix sum setup) ---
zeros = [0] * 6             # pre-allocate 6 slots
print("Zeros:", zeros)

# --- List comprehension ---
squares = [x * x for x in range(6)]
print("Squares:", squares)

# --- Two pointer setup — always initialize like this ---
l, r = 0, len(nums) - 1    # l at left end, r at right end
print(f"Two pointer init: l={l} r={r} | nums[l]={nums[l]} nums[r]={nums[r]}")

# --- Prefix sum setup ---
prefix = [0] * (len(nums) + 1)   # one extra slot — prefix[0] = 0 sentinel
for i, val in enumerate(nums):
    prefix[i + 1] = prefix[i] + val
print("Prefix sum:", prefix)

<a id='3'></a>
## 3. 🛠️ The Core API — Every Operation

---

```
OPERATION              COMPLEXITY   WHAT IT DOES
─────────────────────────────────────────────────────────────
nums[i]                O(1)         Read element at index i
nums[i] = x            O(1)         Write element at index i
nums[-1]               O(1)         Last element (right end)
nums[i:j]              O(k)         Slice — copies k elements
len(nums)              O(1)         Length
nums.append(x)         O(1)*        Add to right end
nums.pop()             O(1)         Remove from right end
nums.pop(0)            O(n)         Remove from left — shifts everything ❌
nums.insert(0, x)      O(n)         Insert at left — shifts everything ❌
nums.insert(i, x)      O(n)         Insert at middle — shifts from i onward
x in nums              O(n)         Linear scan — use set for O(1)
nums.index(x)          O(n)         Find index of first x
nums.sort()            O(n log n)   In-place sort
sorted(nums)           O(n log n)   Returns new sorted list
nums.reverse()         O(n)         In-place reverse
min(nums)/max(nums)    O(n)         Linear scan
sum(nums)              O(n)         Linear scan
enumerate(nums)        O(n)         Yields (index, value) pairs
zip(a, b)              O(n)         Pairs elements from two arrays
─────────────────────────────────────────────────────────────
* amortized

THINGS YOU DO NOT DO:
  ❌  nums.pop(0)      — O(n), use deque.popleft() if you need left removal
  ❌  nums.insert(0,x) — O(n), same reason
  ❌  x in nums        — O(n) for repeated lookups, build a set first
  ❌  nums[i:j] inside a tight loop — copies memory each time
  ❌  mutating input array unless explicitly told it's allowed
```

In [ ]:
nums = [3, 1, 4, 1, 5, 9, 2, 6]

print(f"nums[0]   = {nums[0]}      # first element")
print(f"nums[-1]  = {nums[-1]}      # last element")
print(f"nums[2:5] = {nums[2:5]}  # slice indices 2,3,4")
print(f"len       = {len(nums)}      # length")
print(f"min       = {min(nums)}      # O(n) scan")
print(f"max       = {max(nums)}      # O(n) scan")
print(f"sum       = {sum(nums)}     # O(n) scan")
print()

# enumerate — the clean way to get index + value
print("enumerate:")
for i, val in enumerate(nums[:4]):
    print(f"  i={i}  val={val}")
print()

# zip — pair two arrays element-wise
a = [1, 2, 3]
b = [10, 20, 30]
print("zip:", list(zip(a, b)))
print()

# sort vs sorted
original = [3, 1, 4, 1, 5]
new_sorted = sorted(original)   # original unchanged
original.sort()                 # original modified in place
print(f"sorted()  = {new_sorted}   (new list)")
print(f".sort()   = {original}   (in place)")

<a id='4'></a>
## 4. 🗺️ Decision Map — Two Pointer vs Sliding Window vs Prefix Sum

---

```
  SIGNAL IN THE PROBLEM                        USE THIS
  ──────────────────────────────────────────────────────────────────
  "sorted array" + "pair that sums to target"  Two Pointer (opposite ends)
  "remove duplicates" / "in-place filter"      Two Pointer (same direction)
  "3Sum" / "4Sum" / "pair with constraint"      Two Pointer (sorted + outer loop)
  "trap rainwater" / "container with water"    Two Pointer (opposite ends)

  "fixed window of size k" + "max/min/avg"     Fixed Sliding Window
  "longest substring with constraint"          Variable Sliding Window
  "shortest subarray with sum >= target"       Variable Sliding Window
  "minimum window substring"                   Variable Sliding Window + freq map

  "sum of subarray [i..j]" repeated queries    Prefix Sum
  "subarray sum equals k"                      Prefix Sum + hash map
  "product of array except self"               Prefix product (left) + suffix (right)
  "2D range sum query"                         2D Prefix Sum
  ──────────────────────────────────────────────────────────────────

  QUICK FILTER:
  - Array is SORTED → try Two Pointer first
  - Problem involves a CONTIGUOUS subarray → try Sliding Window
  - Multiple RANGE QUERIES on same array → try Prefix Sum
  - None of the above → consider sorting first, then revisit
```

<a id='5'></a>
## 5. 🧩 Pattern 1: Two Pointer — Opposite Ends — LC 167

---

```
  PROBLEM:
  Given a sorted array, find two indices whose values sum to target.
  Return 1-indexed.

  TRICK:
  Start L at left (smallest), R at right (largest).
  Sum too small → move L right (need bigger left value).
  Sum too big  → move R left  (need smaller right value).
  Sorted order guarantees no valid pair is ever skipped.

  SLOW MOTION on [2, 7, 11, 15], target=9:

  step   L   R   nums[L]  nums[R]  sum   action
   1     0   3     2        15      17   too big  → R-=1
   2     0   2     2        11      13   too big  → R-=1
   3     0   1     2         7       9   MATCH → return [1,2]

  KEY INSIGHT:
  Sorted order makes the decision at each step obvious — no guessing.
  Each step eliminates one element permanently. O(n) total.

  TIME:  O(n) — each pointer moves at most n steps
  SPACE: O(1) — two pointers only
```

In [ ]:
from typing import List

def twoSum(numbers: List[int], target: int) -> List[int]:
    """
    LC 167 — Two Sum II (sorted array)
    Approach: opposite-end two pointers. Move the smaller side inward.
    Args:
        numbers (List[int]): sorted array of integers.
        target  (int): target sum.
    Returns:
        List[int]: 1-indexed pair [l+1, r+1].
    Time:  O(n) — each pointer moves at most n steps total
    Space: O(1) — two pointers, no extra structure
    """
    l, r = 0, len(numbers) - 1   # fingers start at opposite ends

    while l < r:
        s = numbers[l] + numbers[r]
        if s == target:
            return [l + 1, r + 1]   # problem is 1-indexed
        if s > target:
            r -= 1                  # sum too big — shrink from right
        else:
            l += 1                  # sum too small — grow from left

    return []                       # guaranteed a solution exists per constraints


def test_harness(fn):
    tests = [
        ([2, 7, 11, 15], 9,  [1, 2]),
        ([2, 3, 4],      6,  [1, 3]),
        ([-1, 0],       -1,  [1, 2]),
        ([0, 0],         0,  [1, 2]),
        ([1, 2, 3, 4, 5], 9, [4, 5]),
        ([-3, -1, 0, 2], -1, [1, 4]),
        ([1, 3, 5, 7, 9], 8, [1, 4]),
    ]
    passed = 0
    for numbers, target, expected in tests:
        got = fn(numbers, target)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | numbers={numbers} target={target} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")


print(twoSum([2, 7, 11, 15], 9))    # [1, 2]
print(twoSum([2, 3, 4], 6))         # [1, 3]
print(twoSum([-1, 0], -1))          # [1, 2]
test_harness(twoSum)
print("twoSum defined.")

<a id='6'></a>
## 6. 🧩 Pattern 2: Two Pointer — Same Direction / Fast-Slow — LC 26

---

```
  PROBLEM:
  Given a sorted array, remove duplicates in-place.
  Return the count of unique elements. First k elements must be unique.

  TRICK:
  Slow pointer (write_pos) tracks where to write the next unique value.
  Fast pointer (read_pos) scans ahead looking for new unique values.
  When fast finds a value different from slow's last write — copy it over.

  SLOW MOTION on [0,0,1,1,1,2,2,3,3,4]:

  write_pos  read_pos  nums[read]  action
      1          1         0       same as nums[0]=0, skip
      1          2         1       NEW — write to pos 1, write_pos=2
      2          3         1       same, skip
      2          4         1       same, skip
      2          5         2       NEW — write to pos 2, write_pos=3
      3          6         2       same, skip
      3          7         3       NEW — write to pos 3, write_pos=4
      4          8         3       same, skip
      4          9         4       NEW — write to pos 4, write_pos=5
  return 5
  array[:5] = [0, 1, 2, 3, 4]

  KEY INSIGHT:
  Slow is the write head. Fast is the read head.
  Fast races ahead. Slow only moves when it finds something worth keeping.

  TIME:  O(n) — one pass, fast pointer visits each element once
  SPACE: O(1) — in-place, no extra array
```

In [ ]:
def removeDuplicates(nums: List[int]) -> int:
    """
    LC 26 — Remove Duplicates from Sorted Array
    Approach: fast/slow two pointer. Slow = write head. Fast = read head.
    Args:
        nums (List[int]): sorted array, modified in-place.
    Returns:
        int: count of unique elements (k). First k elements are the answer.
    Time:  O(n) — single pass, fast visits each element once
    Space: O(1) — in-place, two pointers only
    """
    if not nums:
        return 0

    write_pos = 1   # slot 0 is always unique — start writing at slot 1

    for read_pos in range(1, len(nums)):
        if nums[read_pos] != nums[write_pos - 1]:   # found a new unique value
            nums[write_pos] = nums[read_pos]         # copy it to write head
            write_pos += 1                           # advance write head
        # else: duplicate — fast moves on, slow stays put

    return write_pos   # number of unique elements written


def test_harness_remove(fn):
    tests = [
        ([1, 1, 2],               2, [1, 2]),
        ([0,0,1,1,1,2,2,3,3,4],  5, [0,1,2,3,4]),
        ([1],                     1, [1]),
        ([1, 2],                  2, [1, 2]),
        ([1, 1, 1],               1, [1]),
    ]
    passed = 0
    for nums, expected_k, expected_vals in tests:
        nums_copy = nums[:]
        k = fn(nums_copy)
        ok = (k == expected_k) and (nums_copy[:k] == expected_vals)
        status = "PASSED" if ok else "FAILED"
        if not ok:
            print(f"{status} | input={nums} | expected k={expected_k} vals={expected_vals} | got k={k} vals={nums_copy[:k]}")
        passed += ok
    print(f"{passed}/{len(tests)} tests passed")


print(removeDuplicates([1, 1, 2]))             # 2
print(removeDuplicates([0,0,1,1,1,2,2,3,3,4])) # 5
test_harness_remove(removeDuplicates)
print("removeDuplicates defined.")

<a id='7'></a>
## 7. 🧩 Pattern 3: Fixed Sliding Window — LC 643

---

```
  PROBLEM:
  Find the maximum average of any contiguous subarray of length k.

  TRICK:
  Build the first window of size k by summing k elements.
  Then slide: add the incoming right element, drop the outgoing left element.
  Track the maximum window sum seen.
  No recomputing the whole window each step — just one add and one subtract.

  SLOW MOTION on [1,12,-5,-6,50,3], k=4:

  window          sum    max_sum
  [1,12,-5,-6]     2       2      initial window
  [12,-5,-6,50]   51      51      add 50, drop 1
  [-5,-6,50,3]    42      51      add 3, drop 12
  answer = 51 / 4 = 12.75

  KEY INSIGHT:
  Each slide is O(1): window_sum += nums[r] - nums[r-k].
  Brute force recomputes k elements per slide = O(n*k).
  Sliding window = O(n). Same answer, fraction of the work.

  TIME:  O(n) — one pass after initial window
  SPACE: O(1) — one running sum variable
```

In [ ]:
def findMaxAverage(nums: List[int], k: int) -> float:
    """
    LC 643 — Maximum Average Subarray I
    Approach: fixed sliding window. Add incoming, drop outgoing. Track max sum.
    Args:
        nums (List[int]): input array.
        k    (int): fixed window size.
    Returns:
        float: maximum average of any k-length subarray.
    Time:  O(n) — one pass, each element added and removed once
    Space: O(1) — one running sum, no extra array
    """
    window_sum = sum(nums[:k])   # build the first window — O(k) once
    max_sum    = window_sum      # best seen so far starts as the first window

    for r in range(k, len(nums)):
        window_sum += nums[r]        # new element slides in from the right
        window_sum -= nums[r - k]    # old element slides out from the left
        max_sum = max(max_sum, window_sum)

    return max_sum / k


def test_harness_avg(fn):
    tests = [
        ([1,12,-5,-6,50,3], 4,  12.75),
        ([5],               1,  5.0),
        ([0,4,0,3,2],       1,  4.0),
        ([0,4,0,3,2],       2,  2.5),
        ([-1],              1, -1.0),
    ]
    passed = 0
    for nums, k, expected in tests:
        got = fn(nums, k)
        ok = abs(got - expected) < 1e-5   # float comparison with tolerance
        status = "PASSED" if ok else "FAILED"
        if not ok:
            print(f"{status} | nums={nums} k={k} | expected={expected} | got={got}")
        passed += ok
    print(f"{passed}/{len(tests)} tests passed")


print(findMaxAverage([1,12,-5,-6,50,3], 4))   # 12.75
print(findMaxAverage([5], 1))                  # 5.0
test_harness_avg(findMaxAverage)
print("findMaxAverage defined.")

<a id='8'></a>
## 8. 🧩 Pattern 4: Variable Sliding Window — LC 3

---

```
  PROBLEM:
  Find the length of the longest substring without repeating characters.

  TRICK:
  Window = [L..R]. Expand R to add new characters.
  When a duplicate appears, shrink from L until the duplicate is gone.
  A hash set tracks what's currently in the window.
  The window always contains only unique characters.

  SLOW MOTION on s = "abcabcbb":

  L   R   char   window set      action           max_len
  0   0    a     {a}             add a               1
  0   1    b     {a,b}           add b               2
  0   2    c     {a,b,c}         add c               3
  0   3    a     {a,b,c}  dup!   shrink L: remove a  3
  1   3    a     {b,c,a}         add a               3
  1   4    b     {b,c,a}  dup!   shrink L: remove b  3
  2   4    b     {c,a,b}         add b               3
  2   5    c     {c,a,b}  dup!   shrink: remove c    3
  3   5    c     {a,b,c}         add c               3
  3   6    b     {a,b,c}  dup!   shrink: remove a    3
  4   6    b     {b,c}    dup!   shrink: remove b    3
  5   6    b     {c,b}           add b               3
  5   7    b     {c,b}    dup!   shrink: remove c    3
  6   7    b     {b}      dup!   shrink: remove b    3
  7   7    b     {b}             add b               3
  answer = 3 ("abc")

  KEY INSIGHT:
  Window contracts from the left only when a constraint is violated.
  Every character enters and exits the set at most once → O(n) total.

  TIME:  O(n) — L and R each move at most n steps
  SPACE: O(min(n, alphabet)) — set holds window contents
```

In [ ]:
def lengthOfLongestSubstring(s: str) -> int:
    """
    LC 3 — Longest Substring Without Repeating Characters
    Approach: variable sliding window with a set to track current window contents.
    Expand R, shrink L on duplicate until window is valid again.
    Args:
        s (str): input string. 0 <= len(s) <= 5*10^4.
    Returns:
        int: length of the longest substring with all unique characters.
    Time:  O(n) — each character enters and exits the set at most once
    Space: O(min(n, k)) — k = alphabet size, set holds window contents
    """
    window = set()    # the current window's character set — no duplicates allowed
    l = 0             # left boundary of window
    max_len = 0

    for r in range(len(s)):
        while s[r] in window:          # duplicate detected — window is invalid
            window.remove(s[l])        # shrink from left until duplicate is gone
            l += 1
        window.add(s[r])               # window is valid — add new right character
        max_len = max(max_len, r - l + 1)

    return max_len


def test_harness_lls(fn):
    tests = [
        ("abcabcbb", 3),
        ("bbbbb",    1),
        ("pwwkew",   3),
        ("",         0),
        ("a",        1),
        ("au",       2),
        ("dvdf",     3),
        ("abcdef",   6),
    ]
    passed = 0
    for s, expected in tests:
        got = fn(s)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | s={s!r} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")


print(lengthOfLongestSubstring("abcabcbb"))  # 3
print(lengthOfLongestSubstring("bbbbb"))     # 1
print(lengthOfLongestSubstring("pwwkew"))    # 3
test_harness_lls(lengthOfLongestSubstring)
print("lengthOfLongestSubstring defined.")

<a id='9'></a>
## 9. 🧩 Pattern 5: Prefix Sum — Subarray Sum Equals K — LC 560

---

```
  PROBLEM:
  Count the number of contiguous subarrays that sum to k.
  Array may contain negatives — sliding window won't work.

  TRICK:
  If prefix[j] - prefix[i] == k, then subarray [i..j-1] sums to k.
  Rearrange: prefix[i] = prefix[j] - k.
  For each j, look up (current_prefix - k) in a hash map.
  How many times have we seen that value before? That many valid subarrays end at j.

  SLOW MOTION on nums=[1,1,1], k=2:

  j   num   prefix   prefix-k   seen map           count
  —    —       0        —        {0:1}               0    init: seen prefix=0 once
  0    1       1       1-2=-1    {0:1}               0    -1 not in seen
  1    1       2       2-2= 0    {0:1, 1:1}          1    0 is in seen → +1
  2    1       3       3-2= 1    {0:1, 1:1, 2:1}     2    1 is in seen → +1
  answer = 2

  KEY INSIGHT:
  The map stores prefix sums seen SO FAR — it catches subarrays that
  started anywhere before j and end exactly at j.
  Seeding {0: 1} handles subarrays that start at index 0.

  TIME:  O(n) — one pass, O(1) hash map lookup each step
  SPACE: O(n) — hash map stores at most n distinct prefix sums
```

In [ ]:
from collections import defaultdict

def subarraySum(nums: List[int], k: int) -> int:
    """
    LC 560 — Subarray Sum Equals K
    Approach: prefix sum + hash map. For each j, count how many prior
    prefix values equal (current_prefix - k). Each match = one valid subarray.
    Args:
        nums (List[int]): array, may contain negatives.
        k    (int): target subarray sum.
    Returns:
        int: count of contiguous subarrays summing to k.
    Time:  O(n) — one pass, O(1) hash map ops
    Space: O(n) — hash map stores prefix sum frequencies
    """
    seen = defaultdict(int)   # prefix_sum -> how many times we've seen it
    seen[0] = 1               # empty prefix (before index 0) counts as one occurrence

    prefix = 0
    count  = 0

    for num in nums:
        prefix += num                    # running prefix sum up to here
        count  += seen[prefix - k]       # how many prior prefixes make a valid subarray
        seen[prefix] += 1                # record this prefix for future lookups

    return count


def test_harness_sub(fn):
    tests = [
        ([1, 1, 1],       2, 2),
        ([1, 2, 3],       3, 2),
        ([1, -1, 1],      1, 3),
        ([0, 0, 0, 0, 0], 0, 15),
        ([1],             1, 1),
        ([1],             0, 0),
        ([-1, -1, 1],     0, 1),
    ]
    passed = 0
    for nums, k, expected in tests:
        got = fn(nums, k)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | nums={nums} k={k} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")


print(subarraySum([1, 1, 1], 2))   # 2
print(subarraySum([1, 2, 3], 3))   # 2
test_harness_sub(subarraySum)
print("subarraySum defined.")

<a id='10'></a>
## 10. 🧩 Pattern 6: Prefix Product — Product Except Self — LC 238

---

```
  PROBLEM:
  Return an array where answer[i] = product of all elements EXCEPT nums[i].
  No division allowed. O(n) time, O(1) extra space.

  TRICK:
  answer[i] = (product of everything LEFT of i) * (product of everything RIGHT of i)

  Pass 1 (left to right): fill answer[i] with left product.
  Pass 2 (right to left): multiply answer[i] by running right product.
  No extra arrays — the output array is the workspace.

  SLOW MOTION on [1, 2, 3, 4]:

  Pass 1 — left products:
  i=0: answer[0] = 1           (nothing to the left)
  i=1: answer[1] = 1*1 = 1
  i=2: answer[2] = 1*2 = 2
  i=3: answer[3] = 2*3 = 6
  answer = [1, 1, 2, 6]

  Pass 2 — multiply by right products:
  right=1
  i=3: answer[3] *= 1   → 6*1=6,   right=1*4=4
  i=2: answer[2] *= 4   → 2*4=8,   right=4*3=12
  i=1: answer[1] *= 12  → 1*12=12, right=12*2=24
  i=0: answer[0] *= 24  → 1*24=24, right=24*1=24
  answer = [24, 12, 8, 6]  ✓

  KEY INSIGHT:
  Two passes using the output array as workspace = O(1) extra space.
  Division would handle zeros wrong — this approach sidesteps the issue.

  TIME:  O(n) — two linear passes
  SPACE: O(1) extra — output array doesn't count per problem statement
```

In [ ]:
def productExceptSelf(nums: List[int]) -> List[int]:
    """
    LC 238 — Product of Array Except Self
    Approach: two passes. Pass 1 fills left products. Pass 2 multiplies in right products.
    Output array is the workspace — no extra O(n) array needed.
    Args:
        nums (List[int]): input array. 2 <= len <= 10^5. No division allowed.
    Returns:
        List[int]: answer[i] = product of all nums except nums[i].
    Time:  O(n) — two linear passes
    Space: O(1) extra — output array excluded per problem statement
    """
    n = len(nums)
    answer = [1] * n

    # Pass 1: fill answer[i] with product of everything to the LEFT of i
    left_product = 1
    for i in range(n):
        answer[i] = left_product       # nothing is to the left of i=0, so 1
        left_product *= nums[i]        # extend left product to include i for next iteration

    # Pass 2: multiply answer[i] by product of everything to the RIGHT of i
    right_product = 1
    for i in range(n - 1, -1, -1):
        answer[i] *= right_product     # bake in the right side
        right_product *= nums[i]       # extend right product for next iteration

    return answer


def test_harness_prod(fn):
    tests = [
        ([1,2,3,4],     [24,12,8,6]),
        ([-1,1,0,-3,3], [0,0,9,0,0]),
        ([2,3],         [3,2]),
        ([1,1,1,1],     [1,1,1,1]),
        ([0,0],         [0,0]),
        ([1,0],         [0,1]),
    ]
    passed = 0
    for nums, expected in tests:
        got = fn(nums)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | nums={nums} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")


print(productExceptSelf([1,2,3,4]))      # [24,12,8,6]
print(productExceptSelf([-1,1,0,-3,3]))  # [0,0,9,0,0]
test_harness_prod(productExceptSelf)
print("productExceptSelf defined.")

<a id='11'></a>
## 11. 🗺️ The Array Pattern Decision Map

---

```
  QUESTION TYPE                          KEY TECHNIQUE              LC PROBLEMS
  ──────────────────────────────────────────────────────────────────────────────
  Pair sum in sorted array               Two Pointer (opposite)     167, 15, 42
  Remove duplicates / in-place filter    Two Pointer (fast/slow)    26, 27, 80
  Container with most water              Two Pointer (opposite)     11
  Trap rainwater                         Two Pointer (opposite)     42

  Max/min/avg of fixed-size window       Fixed Sliding Window       643, 219
  Longest substring with constraint      Variable Sliding Window    3, 424, 76
  Shortest subarray with sum >= k        Variable Sliding Window    209

  Range sum query (repeated)             Prefix Sum                 303, 304
  Subarray sum equals k                  Prefix Sum + hash map      560
  Product except self                    Prefix + suffix product    238
  ──────────────────────────────────────────────────────────────────────────────

  THE SORTING TRICK:
  If the array is NOT sorted and the problem involves pairs or triples,
  sort first (O(n log n)) then apply Two Pointer.
  Sorting + Two Pointer beats brute force O(n²) to O(n log n).

  THE NEGATIVES WARNING:
  Sliding Window assumes adding elements GROWS the sum.
  If the array has NEGATIVES, the window can shrink unexpectedly.
  Switch to Prefix Sum + hash map for arrays with negatives.

  THE IN-PLACE RULE:
  Fast/Slow pointer is always your first move for in-place problems.
  Slow = write head (only advances on a valid value).
  Fast = read head (always advances).
```

<a id='12'></a>
## 12. 📋 Interview Cheat Sheet

---

### When to reach for each pattern:

| Signal in the Problem | Pattern | First Move |
|---|---|---|
| Sorted array + pair/triplet sum | Two Pointer (opposite) | `l, r = 0, len-1` |
| Remove/filter in-place | Two Pointer (fast/slow) | `write = 0` or `write = 1` |
| Fixed window size k | Fixed Sliding Window | `sum(nums[:k])` then slide |
| Longest/shortest substring + constraint | Variable Sliding Window | `l=0, set or dict` |
| Range queries on same array | Prefix Sum | `prefix = [0]*(n+1)` |
| Subarray sum = k (with negatives) | Prefix Sum + hash map | `seen = {0: 1}` |
| Product without division | Prefix + suffix pass | two passes over output array |

---

### The O(1) operations — memorize these:

```python
nums[i]              # read   O(1)
nums[i] = x          # write  O(1)
nums[-1]             # last   O(1)
nums.append(x)       # push right  O(1) amortized
nums.pop()           # pop right   O(1)
len(nums)            # length      O(1)
```

---

### Templates:

```python
# TWO POINTER — OPPOSITE ENDS (sorted array)
l, r = 0, len(nums) - 1
while l < r:
    s = nums[l] + nums[r]
    if s == target: return [l, r]
    if s > target:  r -= 1
    else:           l += 1

# TWO POINTER — FAST/SLOW (in-place filter)
write = 0
for read in range(len(nums)):
    if is_valid(nums[read]):     # define your validity condition
        nums[write] = nums[read]
        write += 1
return write

# FIXED SLIDING WINDOW
window_sum = sum(nums[:k])
best = window_sum
for r in range(k, len(nums)):
    window_sum += nums[r] - nums[r - k]
    best = max(best, window_sum)

# VARIABLE SLIDING WINDOW
l = 0
window = set()    # or dict for freq
best = 0
for r in range(len(s)):
    while constraint_violated(s[r], window):
        window.remove(s[l])
        l += 1
    window.add(s[r])
    best = max(best, r - l + 1)

# PREFIX SUM — RANGE QUERY
prefix = [0] * (len(nums) + 1)
for i, val in enumerate(nums):
    prefix[i+1] = prefix[i] + val
# sum of nums[i..j] = prefix[j+1] - prefix[i]

# PREFIX SUM + HASH MAP — SUBARRAY SUM = K
from collections import defaultdict
seen = defaultdict(int)
seen[0] = 1
prefix = count = 0
for num in nums:
    prefix += num
    count  += seen[prefix - k]
    seen[prefix] += 1
```

---

### Gotchas to not forget:

```
❌  Sliding window on arrays with negatives → use prefix sum + hash map instead
❌  Forget to seed seen={0:1} in subarray sum hash map → misses subarrays from index 0
❌  Two Pointer on unsorted array → sort first
❌  nums.pop(0) or nums.insert(0,x) inside a loop → O(n²) total, use deque
❌  Mutating input array without being told it's ok → always copy if unsure
✅  Prefix sum query: sum[i..j] = prefix[j+1] - prefix[i] (not prefix[j])
✅  Fixed window: window_sum += nums[r] - nums[r-k] is the one-liner slide
✅  Variable window: shrink from left until constraint is satisfied, then record
✅  Fast/slow: slow only moves when it finds something worth keeping
✅  Sort + Two Pointer: O(n log n) beats brute force O(n²) for pair/triplet problems
```

---
## 🎯 Summary Map

```
                            PYTHON ARRAYS
                         (numbered mailbox row)
                         nums[i] = O(1) access

                    ┌──────────────────────────┐
                    │    3 interview patterns   │
                    └───────────┬──────────────┘
                                │
          ┌─────────────────────┼─────────────────────┐
          │                     │                     │
   TWO POINTER           SLIDING WINDOW          PREFIX SUM
   two fingers               a moving frame      shadow array
   on the row                over the row        of totals
          │                     │                     │
    ┌─────┴─────┐         ┌─────┴─────┐         ┌─────┴─────┐
    │           │         │           │         │           │
 OPPOSITE    FAST/SLOW  FIXED k   VARIABLE   RANGE SUM  +HASH MAP
   ENDS      write/read  window    window     O(1) query  subarray=k
    │           │         │           │         │           │
 LC 167      LC 26     LC 643      LC 3      LC 303      LC 560
 LC 11       LC 80     LC 219      LC 424    LC 238      LC 974
 LC 42       LC 27                 LC 76
```

---
*End of Arrays Master Guide — Sean Edition*